# Exploratory Data Analysis
Understanding the dataset to explore how the data is present in the database and if there is a need of creating some aggregated tables that can help with:
* Vendor selection for profitability
* Product pricing optimization

In [1]:
import pandas as pd
import sqlite3

In [2]:
# creating database connection
conn = sqlite3.connect('inventory.db')

In [3]:
# checking tables present in the database
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table'", conn)
tables

,name
0,begin_inventory
1,end_inventory
2,purchase_prices
3,purchases
4,sales


In [4]:
for table in tables['name']:
    print('-'*50, f'{table}', '-'*50)
    print('count of records:', pd.read_sql(f"select count(*) as count from {table}", conn)['count'].values[0])
    display(pd.read_sql(f"select * from {table} limit 5", conn))


-------------------------------------------------- begin_inventory --------------------------------------------------
count of records: 0


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,startDate


-------------------------------------------------- end_inventory --------------------------------------------------
count of records: 0


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,endDate


-------------------------------------------------- purchase_prices --------------------------------------------------
count of records: 0


,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName


-------------------------------------------------- purchases --------------------------------------------------
count of records: 0


,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification


-------------------------------------------------- sales --------------------------------------------------
count of records: 0


,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName


In [5]:
purchases = pd.read_sql_query(
    "SELECT * FROM purchases WHERE VendorNumber = 4466 LIMIT 500",
    conn
)
purchases

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification


In [6]:
purchase_prices = pd.read_sql_query(
    "SELECT * FROM purchase_prices WHERE VendorNumber = 4466 LIMIT 500",
    conn
)
purchase_prices

,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName


In [7]:
vendor_invoice = pd.read_sql_query(
    "SELECT * FROM vendor_invoice WHERE VendorNumber = 4466 LIMIT 500",
    conn
)
vendor_invoice

DatabaseError: Execution failed on sql 'SELECT * FROM vendor_invoice WHERE VendorNumber = 4466 LIMIT 500': no such table: vendor_invoice

In [ ]:
sales = pd.read_sql_query(
    "SELECT * FROM sales WHERE VendorNo = 4466 LIMIT 500",
    conn
)
sales

,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-09,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
1,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-12,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
2,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-15,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
3,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-21,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
4,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-23,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,32_MOUNTMEND_5255,32,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-02-25,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
496,33_HORNSEY_5215,33,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-02-02,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
497,33_HORNSEY_5215,33,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-02-06,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
498,33_HORNSEY_5215,33,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-02-27,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE


In [ ]:
# purchases_summary = pd.read_sql_query("""
#     SELECT Brand, PurchasePrice,
#            SUM(Quantity) as total_qty,
#            SUM(Dollars) as total_dollars
#     FROM purchases
#     WHERE VendorNumber = 4466
#     GROUP BY Brand, PurchasePrice
# """, conn)
# purchases_summary.head()

purchases.groupby(['Brand', 'PurchasePrice'])[['Quantity', 'Dollars']].sum()

,,Quantity,Dollars
Brand,PurchasePrice,,
3140,11.19,458,5125.02
5215,9.41,1368,12872.88
5255,9.35,1521,14221.35


In [ ]:
# sales_summary = pd.read_sql_query("""
#     SELECT Brand,
#            SUM(SalesDollars) as total_sales,
#            SUM(SalesPrice) as total_price,
#            SUM(SalesQuantity) as total_qty
#     FROM sales
#     WHERE VendorNo = 4466
#     GROUP BY Brand
# """, conn)

sales.groupby('Brand')[['SalesDollars','SalesPrice', 'SalesQuantity']].sum()

,SalesDollars,SalesPrice,SalesQuantity
Brand,,,
5215,4598.46,2831.82,354
5255,4845.27,3663.18,373


## Insight
* The purchases table contains actual purchases data, including the purchases date, product purchases by vendor, the ammount paid, and the quantity purchased.
* The purchase price column in derived from the purchase_price table which provides product wise actual and purchase price. The combination of vendor and brand is unique in this table.
* The vendor_invoice table aggregates data from the purchases table, summarizing quantity and dollar amounts, along with and additional column for freight. This table maintains uniqueness based on vendor and PO number.
* The sales table captures actual sales transactions, detailing the brand purchased by vendors, the quantity sold, the selling price and the revenue earned
---
As the data we need for analysis is distributed in different tables, we need to create a summary table containing :
* Purchased transactions made by vendor
* Sales transaction data
* Freight costs for each vendor
* Actual product price from vendors


In [ ]:
# Freight cost
# first check the vendor_invoice columns
vendor_invoice.columns

Index(['VendorNumber', 'VendorName', 'InvoiceDate', 'PONumber', 'PODate',
       'PayDate', 'Quantity', 'Dollars', 'Freight', 'Approval'],
      dtype='str')

In [ ]:
fright_summary = pd.read_sql_query(
    """
    select VendorNumber, sum(Freight) as FrightCost
    from vendor_invoice
    group by VendorNumber
    """,
    conn
)
fright_summary

,VendorNumber,FrightCost
0,2,108.32
1,54,1.92
2,60,1470.08
3,105,249.56
4,200,24.76
...,...,...
121,98450,3424.08
122,99166,520.36
123,172662,713.36
124,173357,810.00


In [ ]:
# purchases prices
# first check the columns
purchases.columns

Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'VendorNumber',
       'VendorName', 'PONumber', 'PODate', 'ReceivingDate', 'InvoiceDate',
       'PayDate', 'PurchasePrice', 'Quantity', 'Dollars', 'Classification'],
      dtype='str')

In [ ]:
purchase_prices.columns

Index(['Brand', 'Description', 'Price', 'Size', 'Volume', 'Classification',
       'PurchasePrice', 'VendorNumber', 'VendorName'],
      dtype='str')

In [ ]:
pd.read_sql_query(
    """
    select 
    p.VendorNumber,
    p.VendorName,
    p.Brand,
    p.PurchasePrice,
    pp.Volume,
    pp.Price as ActualPrice,
    sum(p.Quantity) as TotalPurchaseQuantity,
    sum(p.Dollars) as TotalPurchaseDollars
    from purchases p
    join purchase_prices pp
    on p.Brand = pp.Brand
    where p.PurchasePrice > 0
    group by p.VendorNumber, p.VendorName, p.Brand
    order by TotalPurchaseDollars
    """,
    conn
)

,VendorNumber,VendorName,Brand,PurchasePrice,Volume,ActualPrice,TotalPurchaseQuantity,TotalPurchaseDollars
0,7245,PROXIMO SPIRITS INC.,3065,0.71,50,0.99,8,5.68
1,3960,DIAGEO NORTH AMERICA INC,6127,1.47,200,1.99,8,11.76
2,3924,HEAVEN HILL DISTILLERIES,9123,0.74,50,0.99,16,11.84
3,8004,SAZERAC CO INC,5683,0.39,50,0.49,48,18.72
4,9815,WINE GROUP INC,8527,1.32,750,4.99,16,21.12
...,...,...,...,...,...,...,...,...
10687,3960,DIAGEO NORTH AMERICA INC,3545,21.89,1750,29.99,1131620,24771161.80
10688,3960,DIAGEO NORTH AMERICA INC,4261,16.17,1750,22.99,1660844,26855847.48
10689,17035,PERNOD RICARD USA,8068,18.24,1750,24.99,1515012,27633818.88
10690,4425,MARTIGNETTI COMPANIES,3405,23.19,1750,28.99,1332792,30907446.48


In [ ]:
# sales table
# check column
sales.columns

Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'SalesQuantity',
       'SalesDollars', 'SalesPrice', 'SalesDate', 'Volume', 'Classification',
       'ExciseTax', 'VendorNo', 'VendorName'],
      dtype='str')

In [ ]:
pd.read_sql_query(
    """
    select
    VendorNo,
    Brand,
    sum(SalesDollars) as TotalSalesDollars,
    sum(SalesPrice) as TotalSalesPrice,
    sum(SalesQuantity) as TotalSalesQuantity,
    sum(ExciseTax) as TotalExciseTax
    from sales
    group by VendorNo, Brand
    """, 
    conn
)

,VendorNo,Brand,TotalSalesDollars,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax
0,2,90085,1923.48,850.77,52,5.78
1,2,90609,1799.28,1349.46,72,1.56
2,60,771,1573.95,1139.24,105,82.70
3,60,3979,154387.34,96668.27,9066,16660.85
4,105,2529,899.70,149.95,30,23.60
...,...,...,...,...,...,...
11267,173357,2804,12597.20,6388.58,280,220.66
11268,173357,3666,22790.88,11920.23,912,357.73
11269,173357,3848,371.88,185.94,12,9.42
11270,173357,3909,63624.54,36685.32,2546,2006.38


In [ ]:
import time 
start = time.time()
vendor_sales_summary = pd.read_sql_query("""
WITH FreightSummary AS (
    SELECT
        VendorNumber,
        SUM(Freight) AS FreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
),

PurchaseSummary AS (
    SELECT
        p.VendorNumber,
        p.VendorName,
        p.Brand,
        AVG(p.PurchasePrice) AS PurchasePrice,
        AVG(pp.Price) AS ActualPrice,
        SUM(p.Quantity) AS TotalPurchaseQuantity,
        SUM(p.Dollars) AS TotalPurchaseDollars
    FROM purchases p
    JOIN purchase_prices pp
        ON p.Brand = pp.Brand
        AND p.VendorNumber = pp.VendorNumber
    WHERE p.PurchasePrice > 0
    GROUP BY p.VendorNumber, p.VendorName, p.Brand
),

SalesSummary AS (
    SELECT
        VendorNo,
        Brand,
        SUM(SalesQuantity) AS TotalSalesQuantity,
        SUM(SalesDollars) AS TotalSalesDollars,
        SUM(SalesPrice) AS TotalSalesPrice,
        SUM(ExciseTax) AS TotalExciseTax
    FROM sales
    GROUP BY VendorNo, Brand
)

SELECT
    ps.VendorNumber,
    ps.VendorName,
    ps.Brand,
    ps.PurchasePrice,
    ps.ActualPrice,
    ps.TotalPurchaseQuantity,
    ps.TotalPurchaseDollars,

    COALESCE(ss.TotalSalesQuantity, 0) AS TotalSalesQuantity,
    COALESCE(ss.TotalSalesDollars, 0) AS TotalSalesDollars,
    COALESCE(ss.TotalSalesPrice, 0) AS TotalSalesPrice,
    COALESCE(ss.TotalExciseTax, 0) AS TotalExciseTax,

    COALESCE(fs.FreightCost, 0) AS FreightCost

FROM PurchaseSummary ps
LEFT JOIN SalesSummary ss
    ON ps.VendorNumber = ss.VendorNo
    AND ps.Brand = ss.Brand
LEFT JOIN FreightSummary fs
    ON ps.VendorNumber = fs.VendorNumber

ORDER BY ps.TotalPurchaseDollars DESC
""", conn)
end = time.time()
print('Time taken to execute the query:', end - start, 'seconds')

Time taken to execute the query: 223.9765019416809 seconds


In [ ]:
vendor_sales_summary

,VendorNumber,VendorName,Brand,PurchasePrice,ActualPrice,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost
0,1128,BROWN-FORMAN CORP,1233,26.27,36.99,1181780,31045360.60,366420,13151795.80,1757240.41,673255.44,274406.72
1,4425,MARTIGNETTI COMPANIES,3405,23.19,28.99,1332792,30907446.48,407850,12275569.38,1466470.36,749385.10,579716.96
2,17035,PERNOD RICARD USA,8068,18.24,24.99,1515012,27633818.88,487558,11832688.92,1208051.19,895846.81,495120.88
3,3960,DIAGEO NORTH AMERICA INC,4261,16.17,22.99,1660844,26855847.48,505292,11318944.08,1095595.11,928437.16,1028128.28
4,3960,DIAGEO NORTH AMERICA INC,3545,21.89,29.99,1131620,24771161.80,351566,10920920.34,1422597.68,645964.87,1028128.28
...,...,...,...,...,...,...,...,...,...,...,...,...
10643,9815,WINE GROUP INC,8527,1.32,4.99,16,21.12,15,47.85,32.88,1.65,108401.64
10644,8004,SAZERAC CO INC,5683,0.39,0.49,48,18.72,396,194.04,3.92,20.80,201174.48
10645,3924,HEAVEN HILL DISTILLERIES,9123,0.74,0.99,16,11.84,4,3.96,1.98,0.20,56279.48
10646,3960,DIAGEO NORTH AMERICA INC,6127,1.47,1.99,8,11.76,186,370.14,197.01,39.06,1028128.28


This query generates a vendor wise sales and purchase summary, which is valuable for:  
**Performance Optimization:**  
* The query involves heavy joins and aggregations on large datasets like sales and purchase.
* Storing the pre-aggregation results avoids repated expensive computation.
* Helps in analyzing sales, purchases and pricing for different vendors and brands.
* Future Benefits of Storing this data for faster dashboarding and reporting.
* Instead of running expensive queries each time, dashboards can featch data quickly from vendor_sales_summary

In [ ]:
vendor_sales_summary.dtypes

VendorNumber               int64
VendorName                   str
Brand                      int64
PurchasePrice            float64
ActualPrice              float64
TotalPurchaseQuantity      int64
TotalPurchaseDollars     float64
TotalSalesQuantity         int64
TotalSalesDollars        float64
TotalSalesPrice          float64
TotalExciseTax           float64
FreightCost              float64
dtype: object

In [ ]:
vendor_sales_summary.isnull().sum()

VendorNumber             0
VendorName               0
Brand                    0
PurchasePrice            0
ActualPrice              0
TotalPurchaseQuantity    0
TotalPurchaseDollars     0
TotalSalesQuantity       0
TotalSalesDollars        0
TotalSalesPrice          0
TotalExciseTax           0
FreightCost              0
dtype: int64

In [ ]:
vendor_sales_summary['VendorName'].unique()

<StringArray>
[          'BROWN-FORMAN CORP',       'MARTIGNETTI COMPANIES',
           'PERNOD RICARD USA',    'DIAGEO NORTH AMERICA INC',
             'BACARDI USA INC',     'JIM BEAM BRANDS COMPANY',
         'MAJESTIC FINE WINES',  'ULTRA BEVERAGE COMPANY LLP',
       'STOLI GROUP,(USA) LLC',        'PROXIMO SPIRITS INC.',
 ...
   'AMERICAN SPIRITS EXCHANGE',                    'UNCORKED',
         'BRONCO WINE COMPANY',     'MILTONS DISTRIBUTING CO',
                'TRUETT HURST',         'LAUREATE IMPORTS CO',
     'FANTASY FINE WINES CORP', 'AAPER ALCOHOL & CHEMICAL CO',
      'SILVER MOUNTAIN CIDERS',      'CAPSTONE INTERNATIONAL']
Length: 127, dtype: str

In [ ]:
vendor_sales_summary['VendorName'] = vendor_sales_summary['VendorName'].str.strip()
vendor_sales_summary.fillna(0, inplace=True)

,VendorNumber,VendorName,Brand,PurchasePrice,ActualPrice,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost
0,1128,BROWN-FORMAN CORP,1233,26.27,36.99,1181780,31045360.60,366420,13151795.80,1757240.41,673255.44,274406.72
1,4425,MARTIGNETTI COMPANIES,3405,23.19,28.99,1332792,30907446.48,407850,12275569.38,1466470.36,749385.10,579716.96
2,17035,PERNOD RICARD USA,8068,18.24,24.99,1515012,27633818.88,487558,11832688.92,1208051.19,895846.81,495120.88
3,3960,DIAGEO NORTH AMERICA INC,4261,16.17,22.99,1660844,26855847.48,505292,11318944.08,1095595.11,928437.16,1028128.28
4,3960,DIAGEO NORTH AMERICA INC,3545,21.89,29.99,1131620,24771161.80,351566,10920920.34,1422597.68,645964.87,1028128.28
...,...,...,...,...,...,...,...,...,...,...,...,...
10643,9815,WINE GROUP INC,8527,1.32,4.99,16,21.12,15,47.85,32.88,1.65,108401.64
10644,8004,SAZERAC CO INC,5683,0.39,0.49,48,18.72,396,194.04,3.92,20.80,201174.48
10645,3924,HEAVEN HILL DISTILLERIES,9123,0.74,0.99,16,11.84,4,3.96,1.98,0.20,56279.48
10646,3960,DIAGEO NORTH AMERICA INC,6127,1.47,1.99,8,11.76,186,370.14,197.01,39.06,1028128.28


In [ ]:
vendor_sales_summary['GrossProfit'] = vendor_sales_summary['TotalSalesDollars'] - vendor_sales_summary['TotalPurchaseDollars']
vendor_sales_summary

,VendorNumber,VendorName,Brand,PurchasePrice,ActualPrice,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit
0,1128,BROWN-FORMAN CORP,1233,26.27,36.99,1181780,31045360.60,366420,13151795.80,1757240.41,673255.44,274406.72,-17893564.80
1,4425,MARTIGNETTI COMPANIES,3405,23.19,28.99,1332792,30907446.48,407850,12275569.38,1466470.36,749385.10,579716.96,-18631877.10
2,17035,PERNOD RICARD USA,8068,18.24,24.99,1515012,27633818.88,487558,11832688.92,1208051.19,895846.81,495120.88,-15801129.96
3,3960,DIAGEO NORTH AMERICA INC,4261,16.17,22.99,1660844,26855847.48,505292,11318944.08,1095595.11,928437.16,1028128.28,-15536903.40
4,3960,DIAGEO NORTH AMERICA INC,3545,21.89,29.99,1131620,24771161.80,351566,10920920.34,1422597.68,645964.87,1028128.28,-13850241.46
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10643,9815,WINE GROUP INC,8527,1.32,4.99,16,21.12,15,47.85,32.88,1.65,108401.64,26.73
10644,8004,SAZERAC CO INC,5683,0.39,0.49,48,18.72,396,194.04,3.92,20.80,201174.48,175.32
10645,3924,HEAVEN HILL DISTILLERIES,9123,0.74,0.99,16,11.84,4,3.96,1.98,0.20,56279.48,-7.88
10646,3960,DIAGEO NORTH AMERICA INC,6127,1.47,1.99,8,11.76,186,370.14,197.01,39.06,1028128.28,358.38


In [ ]:
vendor_sales_summary['ProfitMargin'] = (vendor_sales_summary['GrossProfit'] / vendor_sales_summary['TotalSalesDollars']) * 100

In [ ]:
vendor_sales_summary['StockTurnover'] = vendor_sales_summary['TotalSalesQuantity'] / vendor_sales_summary['TotalPurchaseQuantity']

In [ ]:
vendor_sales_summary['SalesToPurchaseRatio'] = vendor_sales_summary['TotalSalesDollars'] / vendor_sales_summary['TotalPurchaseDollars']

In [ ]:
vendor_sales_summary.columns

Index(['VendorNumber', 'VendorName', 'Brand', 'PurchasePrice', 'ActualPrice',
       'TotalPurchaseQuantity', 'TotalPurchaseDollars', 'TotalSalesQuantity',
       'TotalSalesDollars', 'TotalSalesPrice', 'TotalExciseTax', 'FreightCost',
       'GrossProfit', 'ProfitMargin', 'StockTurnover', 'SalesToPurchaseRatio'],
      dtype='str')

In [ ]:
cursor = conn.cursor()

In [ ]:
cursor.execute("""CREATE TABLE vendor_sales_summary (
    VendorNumber INT,
    VendorName VARCHAR(100),
    Brand INT,
               
    PurchasePrice DECIMAL(10,2),
    ActualPrice DECIMAL(10,2),

    TotalPurchaseQuantity INT,
    TotalPurchaseDollars DECIMAL(15,2),
    TotalSalesQuantity INT,
    TotalSalesDollars DECIMAL(15,2),
    TotalSalesPrice DECIMAL(15,2),
    TotalExciseTax DECIMAL(15,2),
    FreightCost DECIMAL(15,2),
    GrossProfit DECIMAL(15,2),
    ProfitMargin DECIMAL(15,2),
    StockTurnover DECIMAL(15,2),
    SalesToPurchaseRatio DECIMAL(15,2),
    PRIMARY KEY (VendorNumber, Brand)
);
""")

In [ ]:
pd.read_sql_query("SELECT * from vendor_sales_summary", conn)

,VendorNumber,VendorName,Brand,PurchasePrice,ActualPrice,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio
0,1128,BROWN-FORMAN CORP,1233,26.27,36.99,1181780,31045360.60,366420,13151795.80,1757240.41,673255.44,274406.72,-17893564.80,-136.054156,0.310058,0.423632
1,4425,MARTIGNETTI COMPANIES,3405,23.19,28.99,1332792,30907446.48,407850,12275569.38,1466470.36,749385.10,579716.96,-18631877.10,-151.780146,0.306012,0.397172
2,17035,PERNOD RICARD USA,8068,18.24,24.99,1515012,27633818.88,487558,11832688.92,1208051.19,895846.81,495120.88,-15801129.96,-133.537948,0.321818,0.428196
3,3960,DIAGEO NORTH AMERICA INC,4261,16.17,22.99,1660844,26855847.48,505292,11318944.08,1095595.11,928437.16,1028128.28,-15536903.40,-137.264601,0.304238,0.421470
4,3960,DIAGEO NORTH AMERICA INC,3545,21.89,29.99,1131620,24771161.80,351566,10920920.34,1422597.68,645964.87,1028128.28,-13850241.46,-126.823024,0.310675,0.440872
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10643,9815,WINE GROUP INC,8527,1.32,4.99,16,21.12,15,47.85,32.88,1.65,108401.64,26.73,55.862069,0.937500,2.265625
10644,8004,SAZERAC CO INC,5683,0.39,0.49,48,18.72,396,194.04,3.92,20.80,201174.48,175.32,90.352505,8.250000,10.365385
10645,3924,HEAVEN HILL DISTILLERIES,9123,0.74,0.99,16,11.84,4,3.96,1.98,0.20,56279.48,-7.88,-198.989899,0.250000,0.334459
10646,3960,DIAGEO NORTH AMERICA INC,6127,1.47,1.99,8,11.76,186,370.14,197.01,39.06,1028128.28,358.38,96.822824,23.250000,31.474490


In [ ]:
vendor_sales_summary.to_sql('vendor_sales_summary', conn, if_exists='replace', index=False)

10648

In [ ]:
vendor_sales_summary.isnull().sum()

VendorNumber             0
VendorName               0
Brand                    0
PurchasePrice            0
ActualPrice              0
TotalPurchaseQuantity    0
TotalPurchaseDollars     0
TotalSalesQuantity       0
TotalSalesDollars        0
TotalSalesPrice          0
TotalExciseTax           0
FreightCost              0
GrossProfit              0
ProfitMargin             0
StockTurnover            0
SalesToPurchaseRatio     0
dtype: int64